소상공인을 위한 아마존 점포 현황판: 카테고리별 매출(번 돈)과 손님 민심(긍/부정 리뷰) 한눈에 보기

In [1]:
import pandas as pd

df = pd.read_csv("./Portfolio/AmazonSales/amazon.csv")

columns = [
    'product_id', 'product_name', 'category',	'discounted_price', 'actual_price', 'discount_percentage',	'rating', 'rating_count', 'review_title',
    'review_content'
]

df = df[columns]

# Replace Indian Rupee to USD
df['discounted_price'] = round(df['discounted_price'].str.replace('₹', '').str.replace(',', '').astype(float)*0.0104762,2)
df['actual_price'] = round(df['actual_price'].str.replace('₹', '').str.replace(',', '').astype(float)*0.0104762, 2)

# Leave only the first depth of Category
df['category'] = df['category'].str.split("|").str[0]

# replace discount_percentage from 'str' to 'int'
df['discount_percentage'] = (df['discount_percentage'].str.replace('%', '').astype(float)).round(2)

# change data type to int
df = df[df['rating'] != '|'] # remove missing value
df['rating_count'] = df['rating_count'].astype(str).str.replace(',', '', regex=False)
df['rating_count'] = pd.to_numeric(df['rating_count'], errors='coerce')
df['rating_count'] = df['rating_count'].fillna(0)
df['rating_count'] = df['rating_count'].astype(int)
df['rating'] = df['rating'].astype(float)

# export
df.to_csv("cleaned_data.csv", index=False)

#df['rating_count'].describe()
# df.isnull().sum() # NaN
# print(df[['rating', 'rating_count']].dtypes)

df[['actual_price', 'discounted_price']].head(5)



KeyboardInterrupt



In [7]:
from collections import Counter
import re

stopwords = {'a', 'the', 'is', 'are', 'was', 'were', 'this', 'https', 'http', 'com', 'www', 'in', 'i', 'you', 'to', 'for', 'and', 'or', 'it', 'with', 'of', 'my'}
word_data = []

# for pid, group in df.groupby('product_id'):
#     text = " ".join(group['review_content'].astype(str)).lower()
#     words = re.findall(r'\w+', text)
#     filtered_words = [w for w in words if w not in stopwords]
#     counts = Counter(filtered_words)
#     #print(filtered_words)

#     for word, count in counts.most_common(20):
#         word_data.append({
#             'product_id' : pid,
#             'word' : word,
#             'count' : count
#         })

for pid, group in df.groupby('product_id'):
    text = " ".join(group['review_content'].astype(str)).lower()
    words = re.findall(r'\w+', text)
    
    # [수정된 핵심 3줄]: 단어를 앞뒤로 2개씩 강제로 이어 붙입니다 (예: not + happy)
    bi_grams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    
    # 불용어가 들어간 쓸데없는 조합(예: is a, in the)은 걸러냅니다
    filtered_words = [bg for bg in bi_grams if not any(stop in bg.split() for stop in stopwords)]
    
    counts = Counter(filtered_words)
    
    # 이 아래 append 코드는 기존과 완전히 똑같습니다!
    for word, count in counts.most_common(20):
        word_data.append({
            'product_id' : pid,
            'word' : word,
            'count' : count
        })
        
final_df = pd.DataFrame(word_data)
final_df.to_csv('keyword.csv', index=False)